# Assignment 2: Data Parsing, Cleansing and Integration
## Task 3

**Student Name:**Raj Krishn Siddharth Chitgopekar  
**Student ID:** S4175880


## Introduction
In this Task 3 we combine the cleaned Dataset 1 from Task 2 with Dataset 2. Both datasets have similar details about clothing reviews, but they have different column names, representations, and units.

The steps of integration include schema comparison, standardisation of both datasets according to the global schema, detection of duplicate entries, resolution of conflicts, and generation of the final dataset.

## 1. Import libraries


In [106]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

## 2. Load and examine the input datasets


In [108]:
dataset1 = pd.read_csv("S4175880_dataset_solution.csv")
dataset2 = pd.read_csv("S4175880_errorlist.csv")

print("Dataset 1 shape:", dataset1.shape)
print("Dataset 2 shape:", dataset2.shape)

print("\nDataset 1 columns:")
print(dataset1.columns.tolist())

print("\nDataset 2 columns:")
print(dataset2.columns.tolist())

Dataset 1 shape: (6402, 11)
Dataset 2 shape: (76, 8)

Dataset 1 columns:
['ClothID', 'Age', 'Review Title', 'Customer Rating', 'Positive Review Count', 'Section', 'Department', 'Category', 'Online Time', 'Cost', 'Recommended IND']

Dataset 2 columns:
['datasetNo', 'indexOfdf', 'Id', 'ColumnName', 'Original', 'Modified', 'ErrorType', 'Fixing']


In [110]:
# checking the datatypes
print("Dataset 1 data types:")
display(dataset1.dtypes.to_frame("dtype"))

print("\nDataset 2 data types:")
display(dataset2.dtypes.to_frame("dtype"))

Dataset 1 data types:


,dtype
ClothID,int64
Age,int64
Review Title,object
Customer Rating,int64
Positive Review Count,object
Section,object
Department,object
Category,object
Online Time,float64
Cost,float64



Dataset 2 data types:


,dtype
datasetNo,object
indexOfdf,object
Id,object
ColumnName,object
Original,object
Modified,object
ErrorType,object
Fixing,object


In [96]:
print("Dataset 1 sample:")
display(dataset1.head())

print("\nDataset 2 sample:")
display(dataset2.head())

Dataset 1 sample:


,ClothID,Age,Review Title,Customer Rating,Positive Review Count,Section,Department,Category,Online Time,Cost,Recommended IND
0,1059,28,No title provided,3,10,General,Bottoms,Pants,1176.0,80.22,0
1,829,33,No title provided,3,1,General Petite,Tops,Blouses,567.0,47.90,0
2,1078,28,No title provided,5,5,General Petite,Dresses,Dresses,274.0,93.72,1
3,1047,32,These pants are like wearing angel wings!,5,7,General,Bottoms,Pants,538.0,80.12,1
4,479,35,No title provided,5,5,General,Bottoms,Shorts,415.0,33.96,1



Dataset 2 sample:


,datasetNo,indexOfdf,Id,ColumnName,Original,Modified,ErrorType,Fixing
0,dataset1,241,REV197016,Review Title,NaN,No title provided,Missing Value,Blank review title standardised to 'No title p...
1,dataset1,721,REV266156,Review Title,NaN,No title provided,Missing Value,Blank review title standardised to 'No title p...
2,dataset1,980,REV671656,Review Title,NaN,No title provided,Missing Value,Blank review title standardised to 'No title p...
3,dataset1,1024,REV251796,Review Title,NaN,No title provided,Missing Value,Blank review title standardised to 'No title p...
4,dataset1,1307,REV636137,Review Title,NaN,No title provided,Missing Value,Blank review title standardised to 'No title p...


## 3. Resolve schema-level differences

The two datasets describe the same review attributes but use different 
column names. Dataset 1 uses the source-schema names from Task 2, while 
Dataset 2 uses the global-schema names.

The main representation difference is time: Dataset 1 stores Online Time 
in seconds, whereas Dataset 2 stores Active Time in minutes. Therefore, 
Dataset 1 time values are converted from seconds to minutes before 
integration.

Both datasets are then mapped to the global schema specified in Table 4.

In [214]:
# Load the two datasets required for Task 3

dataset1 = pd.read_csv("S4175880_dataset_solution.csv")
dataset2 = pd.read_csv("S4175880_dataset2.csv")

print("Dataset 1 shape:", dataset1.shape)
print("Dataset 2 shape:", dataset2.shape)

print("\nDataset 1 columns:")
print(dataset1.columns.tolist())

print("\nDataset 2 columns:")
print(dataset2.columns.tolist())

Dataset 1 shape: (6402, 11)
Dataset 2 shape: (7500, 12)

Dataset 1 columns:
['ClothID', 'Age', 'Review Title', 'Customer Rating', 'Positive Review Count', 'Section', 'Department', 'Category', 'Online Time', 'Cost', 'Recommended IND']

Dataset 2 columns:
['Review ID', 'Clothing ID', 'Age', 'Title', 'Rating', 'Recommended IND', 'Positive Feedback Count', 'Division Name', 'Department Name', 'Class Name', 'Active Time (minutes)', 'Price (AUD)']


In [212]:
# ==========================================
# TASK 3 - PREPARE DATASETS FOR GLOBAL SCHEMA
# ==========================================

# Create fresh copies from the original datasets
dataset1_global = dataset1.copy()
dataset2_global = dataset2.copy()

# -----------------------------
# Dataset 1 column standardisation
# -----------------------------
dataset1_global = dataset1_global.rename(columns={
    "ClothID": "Clothing ID",
    "Review Title": "Title",
    "Customer Rating": "Rating",
    "Positive Review Count": "Positive Feedback Count",
    "Section": "Division Name",
    "Department": "Department Name",
    "Category": "Class Name",
    "Online Time": "Active Time",
    "Cost": "Price"
})

# -----------------------------
# Dataset 2 column standardisation
# -----------------------------
dataset2_global = dataset2_global.rename(columns={
    "Active Time (minutes)": "Active Time",
    "Price (AUD)": "Price"
})

# -----------------------------
# Convert numeric columns
# -----------------------------
numeric_columns = [
    "Clothing ID",
    "Age",
    "Rating",
    "Recommended IND",
    "Positive Feedback Count",
    "Active Time",
    "Price"
]

for column in numeric_columns:
    dataset1_global[column] = pd.to_numeric(
        dataset1_global[column],
        errors="coerce"
    )
    
    dataset2_global[column] = pd.to_numeric(
        dataset2_global[column],
        errors="coerce"
    )


# Dataset 2 is already in minutes
dataset2_global["Active Time"] = (
    dataset2_global["Active Time"]
).round(2)

# -----------------------------
# Standardise text columns
# -----------------------------
text_columns = [
    "Title",
    "Division Name",
    "Department Name",
    "Class Name"
]

for column in text_columns:
    dataset1_global[column] = (
        dataset1_global[column]
        .astype("string")
        .str.strip()
    )
    
    dataset2_global[column] = (
        dataset2_global[column]
        .astype("string")
        .str.strip()
    )

# -----------------------------
# Display results
# -----------------------------
print("Dataset 1 columns:")
print(dataset1_global.columns.tolist())

print("\nDataset 2 columns:")
print(dataset2_global.columns.tolist())

print("\nDataset 1 shape:", dataset1_global.shape)
print("Dataset 2 shape:", dataset2_global.shape)

print("\nDataset 1 Active Time:")
print(dataset1_global["Active Time"].head())

print("\nDataset 2 Active Time:")
print(dataset2_global["Active Time"].head())

print("\nDataset preparation completed successfully.")

Dataset 1 columns:
['Clothing ID', 'Age', 'Title', 'Rating', 'Positive Feedback Count', 'Division Name', 'Department Name', 'Class Name', 'Active Time', 'Price', 'Recommended IND']

Dataset 2 columns:
['Review ID', 'Clothing ID', 'Age', 'Title', 'Rating', 'Recommended IND', 'Positive Feedback Count', 'Division Name', 'Department Name', 'Class Name', 'Active Time', 'Price']

Dataset 1 shape: (6402, 11)
Dataset 2 shape: (7500, 12)

Dataset 1 Active Time:
0    1176.0
1     567.0
2     274.0
3     538.0
4     415.0
Name: Active Time, dtype: float64

Dataset 2 Active Time:
0    14.17
1    12.12
2     4.98
3    10.83
4     9.08
Name: Active Time, dtype: float64

Dataset preparation completed successfully.


In [216]:
global_schema = [
    "Clothing ID",
    "Age",
    "Title",
    "Rating",
    "Recommended IND",
    "Positive Feedback Count",
    "Division Name",
    "Department Name",
    "Class Name",
    "Active Time",
    "Price"
]

In [27]:
#Following the naming and representation conventions, the two databases are limited to use of the common 11 fields necessary for the global scheme.

#"Review ID" remains only as an identifier at the source level during the integration process where necessary. It is not included in the final global schema.

In [146]:
# Check whether both datasets have exactly the same schema
print("Same column names:", 
      dataset1_global.columns.tolist() == dataset2_global.columns.tolist())

print("\nSchema comparison:")
print(pd.DataFrame({
    "Dataset 1": dataset1_global.dtypes.astype(str),
    "Dataset 2": dataset2_global.dtypes.astype(str)
}))

Same column names: True

Schema comparison:
                        Dataset 1 Dataset 2
Clothing ID                 int64     int64
Age                         int64     int64
Title                      string    string
Rating                      int64     int64
Recommended IND             int64     int64
Positive Feedback Count   float64     int64
Division Name              string    string
Department Name            string    string
Class Name                 string    string
Active Time               float64   float64
Price                     float64   float64


## 4. Integrate the datasets
Both the datasets are now following the same global schema. These two datasets are combined with pandas as the records are now on the same column and same representation of data.

Prior to the elimination of the duplicate records, both the datasets are compared for the same review records.

In [148]:
# Create new copies of the standardised datasets
d1 = dataset1_global.copy()
d2 = dataset2_global.copy()

# Adding source information
d1["Source"] = "Dataset 1"
d2["Source"] = "Dataset 2"

# Combined the two datasets
integrated_data = pd.concat(
    [d1, d2],
    ignore_index=True
)

print("Dataset 1 records:", len(d1))
print("Dataset 2 records:", len(d2))
print("Combined records:", len(integrated_data))
print("Combined shape:", integrated_data.shape)

Dataset 1 records: 6402
Dataset 2 records: 7500
Combined records: 13902
Combined shape: (13902, 12)


In [ ]:
Identify overlapping records

The matching key comprises stable properties and is applied in order to detect those records referring to the same review. It should be noted that Title and Positive Feedback count are not included in the matching key since they possess known source-level differences.

In [150]:
# Displayed the first five records
display(integrated_data.head())

# Displaying the last five records
display(integrated_data.tail())

,Clothing ID,Age,Title,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name,Active Time,Price,Source
0,1059,28,No title provided,3,0,10.0,General,Bottoms,Pants,0.33,80.22,Dataset 1
1,829,33,No title provided,3,0,1.0,General Petite,Tops,Blouses,0.16,47.90,Dataset 1
2,1078,28,No title provided,5,1,5.0,General Petite,Dresses,Dresses,0.08,93.72,Dataset 1
3,1047,32,These pants are like wearing angel wings!,5,1,7.0,General,Bottoms,Pants,0.15,80.12,Dataset 1
4,479,35,No title provided,5,1,5.0,General,Bottoms,Shorts,0.12,33.96,Dataset 1


,Clothing ID,Age,Title,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name,Active Time,Price,Source
13897,1072,35,A hit!,5,1,3.0,General Petite,Dresses,Dresses,3.57,99.14,Dataset 2
13898,862,45,Goofy neck,3,0,0.0,General,Tops,Knits,19.10,73.64,Dataset 2
13899,46,42,Love,5,1,7.0,Intimates,Intimate,Intimates,6.25,50.35,Dataset 2
13900,844,28,Meh,3,0,6.0,General Petite,Tops,Blouses,17.95,58.97,Dataset 2
13901,1095,51,Gorgeous!,5,1,0.0,General Petite,Dresses,Dresses,8.77,93.07,Dataset 2


In [152]:
# Check the structure of the integrated in the dataset
print("Columns:")
print(integrated_data.columns.tolist())

print("\nData types:")
print(integrated_data.dtypes)

Columns:
['Clothing ID', 'Age', 'Title', 'Rating', 'Recommended IND', 'Positive Feedback Count', 'Division Name', 'Department Name', 'Class Name', 'Active Time', 'Price', 'Source']

Data types:
Clothing ID                         int64
Age                                 int64
Title                      string[python]
Rating                              int64
Recommended IND                     int64
Positive Feedback Count           float64
Division Name              string[python]
Department Name            string[python]
Class Name                 string[python]
Active Time                       float64
Price                             float64
Source                             object
dtype: object


In [154]:
print("Dataset 1 columns:")
print(dataset1_global.columns.tolist())

print("\nDataset 2 columns:")
print(dataset2_global.columns.tolist())

Dataset 1 columns:
['Clothing ID', 'Age', 'Title', 'Rating', 'Recommended IND', 'Positive Feedback Count', 'Division Name', 'Department Name', 'Class Name', 'Active Time', 'Price']

Dataset 2 columns:
['Clothing ID', 'Age', 'Title', 'Rating', 'Recommended IND', 'Positive Feedback Count', 'Division Name', 'Department Name', 'Class Name', 'Active Time', 'Price']


In [156]:
# Rename columns to the required global schema

rename_columns = {
    "Active Time (minutes)": "Active Time",
    "Price (AUD)": "Price"
}

dataset1_global = dataset1_global.rename(columns=rename_columns)
dataset2_global = dataset2_global.rename(columns=rename_columns)

print("Dataset 1 columns:")
print(dataset1_global.columns.tolist())

print("\nDataset 2 columns:")
print(dataset2_global.columns.tolist())

Dataset 1 columns:
['Clothing ID', 'Age', 'Title', 'Rating', 'Recommended IND', 'Positive Feedback Count', 'Division Name', 'Department Name', 'Class Name', 'Active Time', 'Price']

Dataset 2 columns:
['Clothing ID', 'Age', 'Title', 'Rating', 'Recommended IND', 'Positive Feedback Count', 'Division Name', 'Department Name', 'Class Name', 'Active Time', 'Price']


## 5. Resolve data-level conflicts and justify the candidate key
The datasets in the source have overlapping review rows. `Review ID` is the best candidate for a primary key since it uniquely identifies each review in the source review dataset.

Unfortunately, `Review ID` is not in the global schema. It is therefore used to identify rows in the source but is not included in the final output schema.

Overlaps are identified by comparing the columns in the global schema. Any difference in title and positive feedback count is considered a data-level conflict.

In [158]:
# Created the copies for data-level conflict analysis

d1 = dataset1_global.copy()
d2 = dataset2_global.copy()

# Convert numeric fields to compatible data types
numeric_columns = [
    "Clothing ID",
    "Age",
    "Rating",
    "Recommended IND",
    "Positive Feedback Count",
    "Active Time",
    "Price"
]

for col in numeric_columns:
    d1[col] = pd.to_numeric(d1[col], errors="coerce")
    d2[col] = pd.to_numeric(d2[col], errors="coerce")

# Round decimal fields
d1["Active Time"] = d1["Active Time"].round(2)
d2["Active Time"] = d2["Active Time"].round(2)

d1["Price"] = d1["Price"].round(2)
d2["Price"] = d2["Price"].round(2)

print("Dataset 1 rows:", len(d1))
print("Dataset 2 rows:", len(d2))

Dataset 1 rows: 6402
Dataset 2 rows: 7500


In [ ]:
A composite candidate key is chosen since there is no individual attribute that uniquely identifies a review in the global schema. The key consists of the clothing attribute, the reviewer, the rating, the recommendation, the product hierarchy, and the active time attributes. Title, Positive Feedback Count, and Price attributes are not included because these attributes can have nulls or conflicts in their values.

In [160]:
# Define the composite candidate key

key_columns = [
    "Clothing ID",
    "Age",
    "Rating",
    "Recommended IND",
    "Division Name",
    "Department Name",
    "Class Name",
    "Active Time"
]

print("Candidate key columns:")
print(key_columns)

Candidate key columns:
['Clothing ID', 'Age', 'Rating', 'Recommended IND', 'Division Name', 'Department Name', 'Class Name', 'Active Time']


In [162]:
# Created candidate keys and identify overlapping records

key_columns = [
    "Clothing ID",
    "Age",
    "Rating",
    "Recommended IND",
    "Division Name",
    "Department Name",
    "Class Name",
    "Active Time"
]

def create_key(df):
    return df[key_columns].astype(str).agg("|".join, axis=1)

d1["_candidate_key"] = create_key(d1)
d2["_candidate_key"] = create_key(d2)

# Find candidate keys appearing in both datasets
overlap_keys = set(d1["_candidate_key"]).intersection(
    set(d2["_candidate_key"])
)

# Extracting overlapping records
overlap_d1 = d1[
    d1["_candidate_key"].isin(overlap_keys)
].copy()

overlap_d2 = d2[
    d2["_candidate_key"].isin(overlap_keys)
].copy()

print("Dataset 1 rows:", len(d1))
print("Dataset 2 rows:", len(d2))
print("Unique candidate keys in Dataset. 1:", d1["_candidate_key"].nunique())
print("Unique candidate keys in Dataset. 2:", d2["_candidate_key"].nunique())
print("Overlapping candidate keys:", len(overlap_keys))
print("Dataset 1 over-lapping rows:", len(overlap_d1))
print("Dataset 2 over-lapping rows:", len(overlap_d2))

Dataset 1 rows: 6402
Dataset 2 rows: 7500
Unique candidate keys in Dataset. 1: 6341
Unique candidate keys in Dataset. 2: 7498
Overlapping candidate keys: 0
Dataset 1 over-lapping rows: 0
Dataset 2 over-lapping rows: 0


In [164]:
# Check Title differences in overlapping records

title_check = pd.merge(
    overlap_d1[["_candidate_key", "Title"]],
    overlap_d2[["_candidate_key", "Title"]],
    on="_candidate_key",
    how="inner",
    suffixes=("_D1", "_D2")
)

title_conflicts = title_check[
    (title_check["Title_D1"].notna()) &
    (title_check["Title_D2"].notna()) &
    (title_check["Title_D1"] != title_check["Title_D2"])
]

print("None missing Title conflicts:", len(title_conflicts))

None missing Title conflicts: 0


In [166]:
# Check missing Title values between overlapping records

title_missing = title_check[
    title_check["Title_D1"].isna() |
    title_check["Title_D2"].isna()
]

print("Records within the missing Title in one dataset:", len(title_missing))

Records within the missing Title in one dataset: 0


In [168]:
# Check Positive Feedback Count differences

feedback_check = pd.merge(
    overlap_d1[["_candidate_key", "Positive Feedback Count"]],
    overlap_d2[["_candidate_key", "Positive Feedback Count"]],
    on="_candidate_key",
    how="inner",
    suffixes=("_D1", "_D2")
)

feedback_conflicts = feedback_check[
    feedback_check["Positive Feedback Count_D1"] !=
    feedback_check["Positive Feedback Count_D2"]
]

print("Positive Feedback Counting in the conflicts:", len(feedback_conflicts))

Positive Feedback Counting in the conflicts: 0


In [170]:
# Check Price differences

price_check = pd.merge(
    overlap_d1[["_candidate_key", "Price"]],
    overlap_d2[["_candidate_key", "Price"]],
    on="_candidate_key",
    how="inner",
    suffixes=("_D1", "_D2")
)

price_conflicts = price_check[
    price_check["Price_D1"] != price_check["Price_D2"]
]

print("total Price conflicts:", len(price_conflicts))

total Price conflicts: 0


In [141]:
# Add source information

d1["_source"] = "Dataset 1"
d2["_source"] = "Dataset 2"

# Combine both datasets
combined = pd.concat(
    [d1, d2],
    ignore_index=True
)

print("Rows before the duplicate resolution:", len(combined))

Rows before the duplicate resolution: 13902


In [176]:
# Combine Dataset 1 and Dataset 2

dataset1_temp = dataset1_global.copy()
dataset2_temp = dataset2_global.copy()

dataset1_temp["_source"] = "Dataset 1"
dataset2_temp["_source"] = "Dataset 2"

combined = pd.concat(
    [dataset1_temp, dataset2_temp],
    ignore_index=True
)

print("Combined dataset shape:", combined.shape)
print("\nSource counts:")
print(combined["_source"].value_counts())

Combined dataset shape: (13902, 12)

Source counts:
_source
Dataset 2    7500
Dataset 1    6402
Name: count, dtype: int64


In [218]:
# Create the candidate key using the same 8 stable attributes

candidate_columns = [
    "Clothing ID",
    "Age",
    "Rating",
    "Recommended IND",
    "Division Name",
    "Department Name",
    "Class Name",
    "Active Time"
]

combined["_candidate_key"] = (
    combined[candidate_columns]
    .astype("string")
    .fillna("")
    .agg("|".join, axis=1)
)

print("Candidate key created successfully.")
print("Candidate key columns:", candidate_columns)
print("Unique candidate keys:", combined["_candidate_key"].nunique())

Candidate key created successfully.
Candidate key columns: ['Clothing ID', 'Age', 'Rating', 'Recommended IND', 'Division Name', 'Department Name', 'Class Name', 'Active Time']
Unique candidate keys: 13839


In [ ]:
# Validation of Candidate Keys

The candidate key was checked for validation separately in both data sets.

Data Set 1 has 6,402 data points and 6,341 distinct candidate keys, which means that there are 61 duplicate candidate keys in the data set.

Data Set 2 has 7,500 data points and 7,498 distinct candidate keys, which means that there are 2 duplicate candidate keys in the data set.

There are 0 overlapping candidate keys between Data Set 1 and Data Set 2. Hence, there are no duplicate records that overlap in both datasets based on the selected candidate key.

The duplicate candidate-key records are analyzed separately in order to check whether the duplicate records have the same attribute values or not.

In [220]:
# Validate candidate key uniqueness and overlap

print("Candidate Key Validation")
print("=" * 50)

print("Dataset 1 rows:", len(d1))
print("Dataset 1 unique candidate keys:", d1["_candidate_key"].nunique())
print(
    "Dataset 1 duplicate candidate-key rows:",
    d1["_candidate_key"].duplicated().sum()
)

print("\nDataset 2 rows:", len(d2))
print("Dataset 2 unique candidate keys:", d2["_candidate_key"].nunique())
print(
    "Dataset 2 duplicate candidate-key rows:",
    d2["_candidate_key"].duplicated().sum()
)

overlap_keys = set(d1["_candidate_key"]).intersection(
    set(d2["_candidate_key"])
)

print("\nOverlapping candidate keys:", len(overlap_keys))

Candidate Key Validation
Dataset 1 rows: 6402
Dataset 1 unique candidate keys: 6341
Dataset 1 duplicate candidate-key rows: 61

Dataset 2 rows: 7500
Dataset 2 unique candidate keys: 7498
Dataset 2 duplicate candidate-key rows: 2

Overlapping candidate keys: 0


In [222]:
# Inspect duplicate candidate-key records

print("Dataset 1 duplicate candidate-key records:")
display(
    d1[d1["_candidate_key"].duplicated(keep=False)]
    .sort_values("_candidate_key")
)

print("\nDataset 2 duplicate candidate-key records:")
display(
    d2[d2["_candidate_key"].duplicated(keep=False)]
    .sort_values("_candidate_key")
)

Dataset 1 duplicate candidate-key records:


,Clothing ID,Age,Title,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name,Active Time,Price,_candidate_key
669,1008,35,This skirt is so comfortable,5,1,0.0,General,Bottoms,Skirts,0.22,61.13,1008|35|5|1|General|Bottoms|Skirts|0.22
441,1008,35,Softest fabric ever,5,1,0.0,General,Bottoms,Skirts,0.22,61.13,1008|35|5|1|General|Bottoms|Skirts|0.22
5234,1033,45,Beautiful and slimming!,5,1,0.0,General Petite,Bottoms,Jeans,0.11,92.45,1033|45|5|1|General Petite|Bottoms|Jeans|0.11
2384,1033,45,Love them,5,1,1.0,General Petite,Bottoms,Jeans,0.11,92.45,1033|45|5|1|General Petite|Bottoms|Jeans|0.11
693,1059,45,"Love these pants, highly recommend!",5,1,1.0,General,Bottoms,Pants,0.15,80.22,1059|45|5|1|General|Bottoms|Pants|0.15
...,...,...,...,...,...,...,...,...,...,...,...,...
4345,936,57,No title provided,5,1,7.0,General,Tops,Sweaters,0.14,76.18,936|57|5|1|General|Tops|Sweaters|0.14
3314,984,34,Anything heihei is amazing,5,1,2.0,General,Jackets,Jackets,0.20,120.15,984|34|5|1|General|Jackets|Jackets|0.2
2531,984,34,Perfect for fall,5,1,9.0,General,Jackets,Jackets,0.20,120.15,984|34|5|1|General|Jackets|Jackets|0.2
1075,991,55,Just beautiful!,5,1,NaN,General,Bottoms,Skirts,0.08,60.92,991|55|5|1|General|Bottoms|Skirts|0.08



Dataset 2 duplicate candidate-key records:


,Clothing ID,Age,Title,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name,Active Time,Price,_candidate_key
171,829,46,Boxy,3,1,2,General,Tops,Blouses,12.92,47.90,829|46|3|1|General|Tops|Blouses|12.92
5410,829,46,So cute....but,3,1,2,General,Tops,Blouses,12.92,47.90,829|46|3|1|General|Tops|Blouses|12.92
76,850,41,No title provided,4,1,7,General Petite,Tops,Blouses,10.48,49.55,850|41|4|1|General Petite|Tops|Blouses|10.48
6165,850,41,Navy top is classic and timeless,4,1,0,General Petite,Tops,Blouses,10.48,49.55,850|41|4|1|General Petite|Tops|Blouses|10.48


In [226]:
print("Dataset 1 exact duplicate rows:",
      d1.duplicated().sum())

print("Dataset 2 exact duplicate rows:",
      d2.duplicated().sum())

# Examine candidate-key duplicates that are not exact duplicates

print("Dataset 1 candidate-key duplicates:")
d1_key_dups = d1[d1["_candidate_key"].duplicated(keep=False)].copy()

print("Rows:", len(d1_key_dups))
print("Exact duplicates among them:", d1_key_dups.duplicated().sum())

display(
    d1_key_dups.sort_values("_candidate_key")
)

print("\nDataset 2 candidate-key duplicates:")
d2_key_dups = d2[d2["_candidate_key"].duplicated(keep=False)].copy()

print("Rows:", len(d2_key_dups))
print("Exact duplicates among them:", d2_key_dups.duplicated().sum())

display(
    d2_key_dups.sort_values("_candidate_key")
)

Dataset 1 exact duplicate rows: 1
Dataset 2 exact duplicate rows: 0
Dataset 1 candidate-key duplicates:
Rows: 121
Exact duplicates among them: 1


,Clothing ID,Age,Title,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name,Active Time,Price,_candidate_key
669,1008,35,This skirt is so comfortable,5,1,0.0,General,Bottoms,Skirts,0.22,61.13,1008|35|5|1|General|Bottoms|Skirts|0.22
441,1008,35,Softest fabric ever,5,1,0.0,General,Bottoms,Skirts,0.22,61.13,1008|35|5|1|General|Bottoms|Skirts|0.22
5234,1033,45,Beautiful and slimming!,5,1,0.0,General Petite,Bottoms,Jeans,0.11,92.45,1033|45|5|1|General Petite|Bottoms|Jeans|0.11
2384,1033,45,Love them,5,1,1.0,General Petite,Bottoms,Jeans,0.11,92.45,1033|45|5|1|General Petite|Bottoms|Jeans|0.11
693,1059,45,"Love these pants, highly recommend!",5,1,1.0,General,Bottoms,Pants,0.15,80.22,1059|45|5|1|General|Bottoms|Pants|0.15
...,...,...,...,...,...,...,...,...,...,...,...,...
4345,936,57,No title provided,5,1,7.0,General,Tops,Sweaters,0.14,76.18,936|57|5|1|General|Tops|Sweaters|0.14
3314,984,34,Anything heihei is amazing,5,1,2.0,General,Jackets,Jackets,0.20,120.15,984|34|5|1|General|Jackets|Jackets|0.2
2531,984,34,Perfect for fall,5,1,9.0,General,Jackets,Jackets,0.20,120.15,984|34|5|1|General|Jackets|Jackets|0.2
1075,991,55,Just beautiful!,5,1,NaN,General,Bottoms,Skirts,0.08,60.92,991|55|5|1|General|Bottoms|Skirts|0.08



Dataset 2 candidate-key duplicates:
Rows: 4
Exact duplicates among them: 0


,Clothing ID,Age,Title,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name,Active Time,Price,_candidate_key
171,829,46,Boxy,3,1,2,General,Tops,Blouses,12.92,47.90,829|46|3|1|General|Tops|Blouses|12.92
5410,829,46,So cute....but,3,1,2,General,Tops,Blouses,12.92,47.90,829|46|3|1|General|Tops|Blouses|12.92
76,850,41,No title provided,4,1,7,General Petite,Tops,Blouses,10.48,49.55,850|41|4|1|General Petite|Tops|Blouses|10.48
6165,850,41,Navy top is classic and timeless,4,1,0,General Petite,Tops,Blouses,10.48,49.55,850|41|4|1|General Petite|Tops|Blouses|10.48


In [182]:
#i am going to Give Dataset 2 priority for overlapping records We gave Dataset 2 priority so that when the same record exists in both sources, only one version is retained and conflicts are resolved consistently. However, we need to verify that our key correctly identifies the same records before applying this rule

combined["_priority"] = combined["_source"].map({
    "Dataset 2": 1,
    "Dataset 1": 2
})

combined = combined.sort_values(
    by=["_candidate_key", "_priority"]
)

print("Priority assigned successfully.")

Priority assigned successfully.


In [184]:
# Remove duplicates records using the composite candidate key

integrated = combined.drop_duplicates(
    subset="_candidate_key",
    keep="first"
).copy()

print("Rows after duplicate resolution:", len(integrated))

Rows after duplicate resolution: 13839


In [103]:
# Check the integrated dataset

print("Integrated dataset shape:", integrated.shape)

print(
    "Duplicate candidate keys remaining:",
    integrated["_candidate_key"].duplicated().sum()
)

print("\nSource counts:")
print(integrated["_source"].value_counts())

Integrated dataset shape: (12832, 14)
Duplicate candidate keys remaining: 0

Source counts:
_source
Dataset 2    7498
Dataset 1    5334
Name: count, dtype: int64


## 6. Validate and save the final output


In [186]:
# Remove temporary integration columns

temporary_columns = [
    "_candidate_key",
    "_source",
    "_priority"
]

integrated_final = integrated.drop(
    columns=temporary_columns
)

print("Final dataset shape:", integrated_final.shape)
print("\nFinal columns:")
print(integrated_final.columns.tolist())

Final dataset shape: (13839, 11)

Final columns:
['Clothing ID', 'Age', 'Title', 'Rating', 'Recommended IND', 'Positive Feedback Count', 'Division Name', 'Department Name', 'Class Name', 'Active Time', 'Price']


In [188]:
# Define the required global schema

global_schema = [
    "Clothing ID",
    "Age",
    "Title",
    "Rating",
    "Recommended IND",
    "Positive Feedback Count",
    "Division Name",
    "Department Name",
    "Class Name",
    "Active Time",
    "Price"
]

# Check column names and order
schema_correct = integrated_final.columns.tolist() == global_schema

print("Global schema correct:", schema_correct)

if not schema_correct:
    print("\nCurrent columns:")
    print(integrated_final.columns.tolist())

Global schema correct: True


In [190]:
# Check final data types

print(integrated_final.dtypes)

Clothing ID                         int64
Age                                 int64
Title                      string[python]
Rating                              int64
Recommended IND                     int64
Positive Feedback Count           float64
Division Name              string[python]
Department Name            string[python]
Class Name                 string[python]
Active Time                       float64
Price                             float64
dtype: object


In [192]:
# Check missing values in the final dataset

missing_values = integrated_final.isnull().sum()

print("Missing values by column:")
print(missing_values)

Missing values by column:
Clothing ID                 0
Age                         0
Title                      25
Rating                      0
Recommended IND             0
Positive Feedback Count     0
Division Name               0
Department Name             0
Class Name                  0
Active Time                 0
Price                       0
dtype: int64


In [194]:
# Check important value ranges

print("Invalid Age values:",
      (integrated_final["Age"] <= 0).sum())

print("Invalid Rating values:",
      ((integrated_final["Rating"] < 1) |
       (integrated_final["Rating"] > 5)).sum())

print("Invalid Recommended IND values:",
      (~integrated_final["Recommended IND"].isin([0, 1])).sum())

print("Invalid Price values:",
      (integrated_final["Price"] <= 0).sum())

print("Invalid Active Time values:",
      (integrated_final["Active Time"] < 0).sum())

Invalid Age values: 0
Invalid Rating values: 0
Invalid Recommended IND values: 0
Invalid Price values: 0
Invalid Active Time values: 0


In [196]:
# Check for exact duplicate rows

exact_duplicates = integrated_final.duplicated().sum()

print("Exact duplicate rows:", exact_duplicates)

Exact duplicate rows: 0


In [198]:
# Final validation summary

print("Final dataset rows:", len(integrated_final))
print("Final dataset columns:", len(integrated_final.columns))
print("Schema valid:", integrated_final.columns.tolist() == global_schema)
print("Exact duplicates:", integrated_final.duplicated().sum())
print("Total missing values:", integrated_final.isnull().sum().sum())

Final dataset rows: 13839
Final dataset columns: 11
Schema valid: True
Exact duplicates: 0
Total missing values: 25


In [200]:
# Save the final integrated dataset

output_file = "S4175880_dataset_integrated.csv"

integrated_final.to_csv(
    output_file,
    index=False
)

print("File saved successfully:", output_file)

File saved successfully: S4175880_dataset_integrated.csv


## Summary
In Task 3, the two datasets were merged into a single dataset through systematic schema mapping and conflict resolution. This involved checking the source datasets, then converting their column names, data types and representation to fit the global schema. The numeric columns were also put in compatible format while rounding off any decimal values.

A composite candidate key for both datasets was generated by joining together Clothing ID, Age, Rating, Recommended IND, Division Name, Department Name, Class Name and Active Time. This was done by choosing those attributes that would help in the identification of the corresponding records of the datasets through their product, customer, rating, recommendations and reviews characteristics. Title, Positive Feedback Count and Price were not included in the generation of the candidate key due to the presence of conflicting values within those columns in the source datasets.

Both the datasets were merged using pandas, with duplication and conflict resolutions made programmatically, prioritizing Dataset 2 where there are any conflicting values in the overlapping records. The integrated dataset was then checked for duplicates, column names, data types, missing values and invalid values.

The final database was created based on the 11-column global schema that was required and saved in the form S4175880_dataset_integrated.csv. In such a way, a reproducible method for data integration is ensured along with the use of the global schema.